# Pre-process global subduction observations

**Author:** Haoyuan Li

This notebook prepares two global observational views for prescribed-velocity research: the Slab2 slab-surface depth compilation and the Global Centroid Moment Tensor (GCMT) intermediate-depth and deep-focus earthquake catalog. The maps reproduce the global examples developed in gdmate's `slab2.ipynb` and `global_deep_earthquakes.ipynb`.

Slab data are from Hayes (2018), [Slab2 data release](https://doi.org/10.5066/F7PV6JNV). Earthquake data are from the [Global CMT catalog](https://www.globalcmt.org/CMTfiles.html); the event selection follows Figure 3a of Zhan (2020), [*Mechanisms and Implications of Deep Earthquakes*](https://doi.org/10.1146/annurev-earth-053018-060314).


## Configure the data source

By default, the notebook reuses the datasets already cached by the gdmate cookbooks. The gdmate repository is discovered from the installed package, so its absolute location is not hard-coded.

For an independent copy, set `use_existing_gdmate_data = False`. With `download_missing_data = True`, missing files are downloaded into this notebook's ignored `data` directory. With both switches false, an existing local cache is used without network access. Downloading into the shared gdmate cache is deliberately disallowed.


In [ ]:
import gzip
import math
import os
import sys
from datetime import datetime
from pathlib import Path
from tempfile import NamedTemporaryFile
from urllib.request import Request, urlopen

working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "hamageolib").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import cartopy.crs as ccrs
import gdmate
import matplotlib.pyplot as plt
import numpy as np
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from cmcrameri import cm as cmc
from gdmate.plate_boundaries import (
    load_plate_boundaries,
    plate_boundary_segments,
)
from gdmate.slab2 import (
    download_slab2,
    list_slab2_regions,
    load_earth_relief,
    load_seafloor_age,
    load_slab2_depth,
)
from hamageolib.utils.plot_helper import scale_matplotlib_params
from matplotlib import patheffects, rcdefaults
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LightSource, Normalize
from matplotlib.lines import Line2D

gdmate_repository_root = Path(gdmate.__file__).resolve().parent.parent

use_existing_gdmate_data = True
download_missing_data = False
overwrite_downloads = False
network_timeout_s = 120
plot_scaling_factor = 1.0
legend_font_scaling_multiplier = 1.0
line_width_scaling_multiplier = 1.0
plot_element_scaling_factor = (
    plot_scaling_factor * line_width_scaling_multiplier
)


def configure_matplotlib_plot(font_scale):
    if font_scale <= 0:
        raise ValueError("The figure font scale must be positive")
    rcdefaults()
    scale_matplotlib_params(
        plot_scaling_factor,
        font_scaling_multiplier=font_scale,
        legend_font_scaling_multiplier=legend_font_scaling_multiplier,
        line_width_scaling_multiplier=line_width_scaling_multiplier,
    )
    plt.rcParams.update({
        "font.family": "Times New Roman",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

if use_existing_gdmate_data and download_missing_data:
    raise ValueError(
        "Disable use_existing_gdmate_data before downloading a local copy."
    )

if use_existing_gdmate_data:
    data_root = gdmate_repository_root / "notebooks/data"
else:
    data_root = repository_root / "notebooks/research/prescribe_velocity/data"

slab_data_directory = data_root / "slab2"
gcmt_data_directory = data_root / "gcmt"
catalog_file = gcmt_data_directory / "jan76_dec25.ndk.gz"
catalog_url = (
    "https://www.ldeo.columbia.edu/~gcmt/projects/CMT/catalog/"
    "jan76_dec25.ndk.gz"
)
print(f"Data source: {data_root}")


## Locate or download the datasets

The download mode retrieves every Slab2 depth region from the official USGS directory and the maintained compressed GCMT catalog. The background seafloor-age, relief and PB2002 files are requested when the Slab2 map is made. Existing files are reused unless `overwrite_downloads` is enabled.


In [ ]:
def download_file(url, destination, *, overwrite=False, timeout=120):
    destination = Path(destination)
    if destination.is_file() and not overwrite:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = Request(url, headers={"User-Agent": "hamageolib-preprocess/0.1"})
    temporary = None
    try:
        with urlopen(request, timeout=timeout) as response, NamedTemporaryFile(
            dir=destination.parent, suffix=".part", delete=False
        ) as stream:
            temporary = Path(stream.name)
            while block := response.read(1024 * 1024):
                stream.write(block)
        with gzip.open(temporary, "rb") as stream:
            stream.read(1)
        os.replace(temporary, destination)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
    return destination


if download_missing_data:
    slab_regions = list_slab2_regions(field="depth", timeout=network_timeout_s)
    for slab_region in slab_regions:
        download_slab2(
            slab_region, slab_data_directory, fields=("depth",),
            overwrite=overwrite_downloads, timeout=network_timeout_s,
        )
    download_file(
        catalog_url, catalog_file, overwrite=overwrite_downloads,
        timeout=network_timeout_s,
    )

slab_depth_files = sorted(slab_data_directory.glob("*_slab2_dep_*.grd"))
if not slab_depth_files:
    raise FileNotFoundError(
        f"No Slab2 depth grids in {slab_data_directory}. "
        "Select the gdmate cache or enable downloads."
    )
if not catalog_file.is_file():
    raise FileNotFoundError(
        f"{catalog_file}: select the gdmate cache or enable downloads."
    )
print(f"Slab2 depth regions: {len(slab_depth_files)}")
print(f"GCMT catalog: {catalog_file}")


## Plot global Slab2 depth

The uppercase labels plotted on the map are the three-letter codes used by the original Slab2 files. Edit `slab_abbreviations_to_hide` to suppress selected labels or `slab_abbreviations_to_highlight` to draw selected labels in red. Set `global_slab_font_scale` to scale every font in this figure.

| Code | Slab2 region | Code | Slab2 region | Code | Slab2 region |
| --- | --- | --- | --- | --- | --- |
| ALU | Aleutians | CAL | Calabria | CAM | Central America |
| CAR | Caribbean | CAS | Cascadia | COT | Cotabato |
| HAL | Halmahera | HEL | Hellenic | HIM | Himalaya |
| HIN | Hindu Kush | IZU | Izu-Bonin | KER | Kermadec |
| KUR | Kuril | MAK | Makran | MAN | Manila |
| MUE | Muertos | PAM | Pamir | PHI | Philippines |
| PNG | New Guinea | PUY | Puysegur | RYU | Ryukyu |
| SAM | South America | SCO | Scotia | SOL | Solomon Islands |
| SUL | Sulawesi | SUM | Sumatra/Java | VAN | Vanuatu |

This reproduces the global view in gdmate's `slab2.ipynb`: all regional Slab2 depth grids share one positive-down depth scale, above a grayscale seafloor-age and subdued relief background. PB2002 plate boundaries provide tectonic context. The map is Pacific-centred and limited to 65°S–65°N.


In [ ]:
global_slab_font_scale = 1.0
slab_output_directory = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/global_DE"
)
show_slab_abbreviations = True
slab_abbreviations_to_hide = ["COT"]
slab_abbreviations_to_highlight = ["KER", "IZU", "SUM", "KUR", "SAM", "SOL", "HAL"]
slab_label_offsets = {
    "ALU": (0, -5),
    "CAS": (-5, 0),
    "CAM": (0, -5),
    "CAR": (10, -2),
    "IZU": (5, 0),
    "RYU": (2, -3),
    "MAN": (-5, 0),
    "COT": (-7, 1),
    "SUM": (0, -5),
    "HAL": (9, 2),
    "PNG": (0, 3),
    "SOL": (0, -3),
    "VAN": (-5, 0),
    "KER": (5, 0),
    "PUY": (-5, 0),
    "SCO": (7, 0),
    "HIM": (0, -3),
    "HIN": (-2, -3),
    "MAK": (0, -7),
    "HEL": (2, -5),
    "CAL": (0, -5),
    "MUE": (0, 5),
    "PAM": (4, 3),
    "PHI": (6, 1),
    "SUL": (-8, 0),
}
slab_region_names = {
    "ALU": "Aleutians",
    "CAL": "Calabria",
    "CAM": "Central America",
    "CAR": "Caribbean",
    "CAS": "Cascadia",
    "COT": "Cotabato",
    "HAL": "Halmahera",
    "HEL": "Hellenic",
    "HIM": "Himalaya",
    "HIN": "Hindu Kush",
    "IZU": "Izu-Bonin",
    "KER": "Kermadec",
    "KUR": "Kuril",
    "MAK": "Makran",
    "MAN": "Manila",
    "MUE": "Muertos",
    "PAM": "Pamir",
    "PHI": "Philippines",
    "PNG": "New Guinea",
    "PUY": "Puysegur",
    "RYU": "Ryukyu",
    "SAM": "South America",
    "SCO": "Scotia",
    "SOL": "Solomon Islands",
    "SUL": "Sulawesi",
    "SUM": "Sumatra/Java",
    "VAN": "Vanuatu",
}
slab_output_directory.mkdir(parents=True, exist_ok=True)
print(f"Slab2 figure directory: {slab_output_directory}")


In [ ]:
def create_global_map(
    title, relief, plate_boundaries, *, seafloor_age=None, boundary_zorder=4,
    display_region=None, projection_center=None, major_spacing=(60, 30),
    minor_spacing=(30, 15),
):
    if projection_center is None:
        projection_center = central_longitude
    figure = plt.figure(
        figsize=(13 * plot_scaling_factor, 5.5 * plot_scaling_factor)
    )
    axis = figure.add_subplot(
        1, 1, 1, projection=ccrs.PlateCarree(central_longitude=projection_center)
    )
    if display_region is None:
        axis.set_xlim(-180, 180)
        axis.set_ylim(*latitude_bounds)
    else:
        axis.set_extent(display_region, crs=ccrs.PlateCarree())

    relief_values = relief.values
    illumination = LightSource(azdeg=315, altdeg=45).hillshade(
        relief_values, vert_exag=4, dx=18_500, dy=18_500,
    )
    normalized_relief = Normalize(vmin=-8000, vmax=8000, clip=True)(
        relief_values
    )
    terrain_gray = 0.5 + relief_contrast * (normalized_relief - 0.5)
    shaded_terrain = terrain_gray + (
        relief_contrast * 0.2 * (illumination - 0.5)
    )
    if seafloor_age is None:
        background_gray = shaded_terrain
    else:
        age_values = seafloor_age.values
        age_gray = 1.0 - np.clip(age_values, 0, 180) / 180
        background_gray = np.where(
            np.isfinite(age_values), age_gray, shaded_terrain,
        )
    background_rgb = np.repeat(
        np.clip(background_gray, 0, 1)[..., np.newaxis], 3, axis=2,
    )

    transform = ccrs.PlateCarree()
    axis.imshow(
        background_rgb, extent=map_region, origin="lower", transform=transform,
        interpolation="bilinear", zorder=0,
    )
    axis.gridlines(
        xlocs=np.arange(-180, 181, minor_spacing[0]),
        ylocs=np.arange(-90, 91, minor_spacing[1]),
        linewidth=0.2 * plot_element_scaling_factor, color="0.25",
        alpha=0.25, linestyle="--", zorder=1,
    )
    coordinate_gridlines = axis.gridlines(
        draw_labels=True, xlocs=np.arange(-180, 181, major_spacing[0]),
        ylocs=np.arange(-90, 91, major_spacing[1]),
        linewidth=0.35 * plot_element_scaling_factor, color="0.25",
        alpha=0.45, linestyle="--", zorder=1,
    )
    coordinate_gridlines.top_labels = False
    coordinate_gridlines.right_labels = False
    coordinate_gridlines.xformatter = LongitudeFormatter(
        zero_direction_label=True, dateline_direction_label=True,
    )
    coordinate_gridlines.yformatter = LatitudeFormatter()
    for segment in plate_boundary_segments(plate_boundaries):
        axis.plot(
            segment[:, 0], segment[:, 1], color="black",
            linewidth=0.55 * plot_element_scaling_factor,
            transform=transform, zorder=boundary_zorder,
        )
    axis.set_title(title)
    return figure, axis, transform


plot_global_slab = True
central_longitude = 180
latitude_bounds = (-65, 65)
map_region = [
    central_longitude - 180, central_longitude + 180, *latitude_bounds,
]
relief_contrast = 0.35

if plot_global_slab:
    configure_matplotlib_plot(global_slab_font_scale)
    seafloor_age = load_seafloor_age(
        map_region, slab_data_directory, resolution="10m",
        download=download_missing_data,
    )
    relief = load_earth_relief(
        map_region, slab_data_directory, resolution="10m",
        download=download_missing_data,
    )
    plate_boundaries = load_plate_boundaries(
        slab_data_directory, download=download_missing_data,
        timeout=network_timeout_s,
    )
    slab_codes = [
        path.name.split("_slab2_", maxsplit=1)[0].upper()
        for path in slab_depth_files
    ]
    unknown_slab_codes = set(slab_codes) - slab_region_names.keys()
    if unknown_slab_codes:
        raise ValueError(
            f"Missing Slab2 region names for {sorted(unknown_slab_codes)}"
        )
    hidden_slab_codes = {
        code.upper() for code in slab_abbreviations_to_hide
    }
    highlighted_slab_codes = {
        code.upper() for code in slab_abbreviations_to_highlight
    }
    configured_slab_codes = hidden_slab_codes | highlighted_slab_codes
    unknown_configured_codes = configured_slab_codes - slab_region_names.keys()
    if unknown_configured_codes:
        raise ValueError(
            "Unknown slab abbreviations in the label configuration: "
            f"{sorted(unknown_configured_codes)}"
        )
    conflicting_slab_codes = hidden_slab_codes & highlighted_slab_codes
    if conflicting_slab_codes:
        raise ValueError(
            "Slab abbreviations cannot be both hidden and highlighted: "
            f"{sorted(conflicting_slab_codes)}"
        )
    slab_grids = [load_slab2_depth(path) for path in slab_depth_files]
    maximum_slab_depth = max(
        100,
        math.ceil(max(float(grid.max()) for grid in slab_grids) / 100) * 100,
    )
    slab_normalization = Normalize(vmin=0, vmax=maximum_slab_depth)
    slab_figure, slab_axis, transform = create_global_map(
        "Global Slab2 depth and seafloor age", relief, plate_boundaries,
        seafloor_age=seafloor_age,
    )
    for slab_code, slab_grid in zip(slab_codes, slab_grids, strict=True):
        latitude_name, longitude_name = slab_grid.dims
        slab_axis.pcolormesh(
            slab_grid[longitude_name], slab_grid[latitude_name], slab_grid.values,
            cmap=cmc.batlow, norm=slab_normalization, shading="auto",
            transform=transform, rasterized=True, zorder=3,
        )
        if show_slab_abbreviations and slab_code not in hidden_slab_codes:
            longitudes, latitudes = np.meshgrid(
                slab_grid[longitude_name].values,
                slab_grid[latitude_name].values,
            )
            valid_cells = np.isfinite(slab_grid.values)
            longitude_offset, latitude_offset = slab_label_offsets.get(
                slab_code, (0, 0)
            )
            label = slab_axis.text(
                float(np.median(longitudes[valid_cells])) + longitude_offset,
                float(np.median(latitudes[valid_cells])) + latitude_offset,
                slab_code,
                fontsize=0.7 * plt.rcParams["font.size"],
                fontweight="bold", ha="center", va="center",
                color=(
                    "red" if slab_code in highlighted_slab_codes else "black"
                ),
                transform=transform, zorder=5,
            )
            label.set_path_effects([
                patheffects.withStroke(
                    linewidth=2 * plot_element_scaling_factor,
                    foreground="white",
                )
            ])

    slab_figure.subplots_adjust(bottom=0.29)
    slab_colorbar_axis = slab_figure.add_axes([0.28, 0.16, 0.48, 0.025])
    slab_colorbar = slab_figure.colorbar(
        ScalarMappable(norm=slab_normalization, cmap=cmc.batlow),
        cax=slab_colorbar_axis, orientation="horizontal",
    )
    slab_colorbar.set_label("Slab surface depth (km)")
    age_colorbar_axis = slab_figure.add_axes([0.28, 0.055, 0.48, 0.025])
    age_colorbar = slab_figure.colorbar(
        ScalarMappable(norm=Normalize(vmin=0, vmax=180), cmap="gray_r"),
        cax=age_colorbar_axis, orientation="horizontal",
    )
    age_colorbar.set_label("Seafloor age (Ma)")
    for suffix in ("png", "pdf"):
        output_file = slab_output_directory / f"global_slab2_depth.{suffix}"
        slab_figure.savefig(output_file, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_file}")
    print(f"Plotted {len(slab_depth_files)} Slab2 regions")
    plt.show()


## Plot intermediate-depth and deep-focus earthquakes

The maintained GCMT aggregate is filtered to 1976–2025. Intermediate-depth events span 70–300 km and deep-focus events are at least 300 km deep. No additional magnitude threshold is imposed. Symbol diameter scales as $M_0^{1/3}$, an estimate of rupture dimension under a constant-stress-drop assumption; the size legend maps selected $M_w$ values to their actual plotted sizes. Set `global_earthquake_font_scale` to scale every font in this figure.


In [ ]:
global_earthquake_font_scale = 1.0
earthquake_output_directory = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/global_DE"
)
earthquake_output_directory.mkdir(parents=True, exist_ok=True)
print(f"Earthquake figure directory: {earthquake_output_directory}")


In [ ]:
catalog_dtype = np.dtype([
    ("event_id", "U16"),
    ("date", "datetime64[D]"),
    ("latitude", "f8"),
    ("longitude", "f8"),
    ("depth_km", "f8"),
    ("moment_dyne_cm", "f8"),
    ("magnitude_mw", "f8"),
])


def read_gcmt_ndk(path):
    with gzip.open(path, "rt", encoding="ascii") as stream:
        lines = stream.read().splitlines()
    if len(lines) % 5:
        raise ValueError("Expected five lines per GCMT NDK event")

    records = []
    for offset in range(0, len(lines), 5):
        hypocenter, identity, centroid, tensor, axes = lines[offset:offset + 5]
        event_date = datetime.strptime(hypocenter[5:15], "%Y/%m/%d").date()
        centroid_fields = centroid.split()
        latitude = float(centroid_fields[3])
        longitude = float(centroid_fields[5])
        depth_km = float(centroid_fields[7])
        exponent = int(tensor[:2])
        scalar_moment = float(axes.split()[10]) * 10.0**exponent
        magnitude = (2.0 / 3.0) * (math.log10(scalar_moment) - 16.1)
        records.append((
            identity[:16].strip(), event_date, latitude, longitude, depth_km,
            scalar_moment, magnitude,
        ))
    return np.array(records, dtype=catalog_dtype)


catalog = read_gcmt_ndk(catalog_file)
date_mask = (
    (catalog["date"] >= np.datetime64("1976-01-01"))
    & (catalog["date"] <= np.datetime64("2025-12-31"))
)
intermediate = catalog[
    date_mask
    & (catalog["depth_km"] >= 70.0)
    & (catalog["depth_km"] < 300.0)
]
deep_focus = catalog[date_mask & (catalog["depth_km"] >= 300.0)]
selected = np.concatenate((intermediate, deep_focus))
print(f"Intermediate-depth events: {intermediate.size:,}")
print(f"Deep-focus events: {deep_focus.size:,}")
print(
    f"Observed Mw range: {selected['magnitude_mw'].min():.2f}–"
    f"{selected['magnitude_mw'].max():.2f}"
)


In [ ]:
def marker_areas(moment_dyne_cm):
    diameter_points = (
        0.9 * plot_element_scaling_factor
        * (moment_dyne_cm / 1.0e24) ** (1.0 / 3.0)
    )
    return diameter_points**2


plot_deep_earthquakes = True
legend_magnitudes = np.array([5.0, 6.0, 7.0, 8.0])

if plot_deep_earthquakes:
    configure_matplotlib_plot(global_earthquake_font_scale)
    earthquake_relief = load_earth_relief(
        map_region, slab_data_directory, resolution="10m",
        download=download_missing_data,
    )
    earthquake_boundaries = load_plate_boundaries(
        slab_data_directory, download=download_missing_data,
        timeout=network_timeout_s,
    )
    figure, axis, transform = create_global_map(
        "GCMT intermediate-depth and deep-focus earthquakes, 1976–2025",
        earthquake_relief, earthquake_boundaries, boundary_zorder=2,
    )

    axis.scatter(
        intermediate["longitude"], intermediate["latitude"],
        s=marker_areas(intermediate["moment_dyne_cm"]),
        color="#2878b5", alpha=0.65, linewidths=0,
        transform=transform, zorder=3,
    )
    axis.scatter(
        deep_focus["longitude"], deep_focus["latitude"],
        s=marker_areas(deep_focus["moment_dyne_cm"]),
        color="#d73027", alpha=0.7, linewidths=0,
        transform=transform, zorder=3,
    )
    depth_legend_handles = [
        Line2D(
            [], [], marker="o", linestyle="none",
            markersize=6 * plot_element_scaling_factor,
            markerfacecolor="#2878b5", markeredgewidth=0,
            label="70–300 km",
        ),
        Line2D(
            [], [], marker="o", linestyle="none",
            markersize=6 * plot_element_scaling_factor,
            markerfacecolor="#d73027", markeredgewidth=0, label="≥300 km",
        ),
    ]
    depth_legend = axis.legend(
        handles=depth_legend_handles, loc="lower left", frameon=True,
    )
    axis.add_artist(depth_legend)

    legend_moments = 10.0 ** (1.5 * legend_magnitudes + 16.1)
    magnitude_legend_handles = [
        Line2D(
            [], [], marker="o", linestyle="none",
            markersize=float(np.sqrt(area)), markerfacecolor="0.35",
            markeredgewidth=0, label=f"$M_w$ {magnitude:.1f}",
        )
        for magnitude, area in zip(
            legend_magnitudes, marker_areas(legend_moments), strict=True,
        )
    ]
    axis.legend(
        handles=magnitude_legend_handles, title="Marker size",
        loc="lower right", frameon=True, labelspacing=1.0,
    )
    for suffix in ("png", "pdf"):
        output_file = earthquake_output_directory / (
            f"global_deep_earthquakes.{suffix}"
        )
        figure.savefig(output_file, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_file}")
    plt.show()


## Plot one Slab2 region and its earthquake catalog

Select any Slab2 model with its three-letter code. Each code has a manually chosen display range that includes the complete regional slab with geographic context. The first figure shows the selected Slab2 surface using the same depth, seafloor-age, relief, and plate-boundary styles as the global map. The second figure uses the same map frame for GCMT intermediate-depth and deep-focus earthquakes whose centroid locations fall inside the valid Slab2 grid footprint.

The regional output therefore supports visual comparison with the global figures while preventing nearby earthquakes assigned to another slab footprint from entering the selected catalog. Set `regional_slab_font_scale` and `regional_earthquake_font_scale` independently to scale every font in the two figures.


In [ ]:
regional_slab_code = "HAL"
regional_slab_font_scale = 1.0
regional_earthquake_font_scale = 1.0
regional_output_directory = Path(
    "/mnt/lochy/ASPECT_DATA/MOW/test_prescribe/global_DE"
)
regional_map_regions = {
    "ALU": [158, 230, 42, 70],
    "CAL": [5, 27, 27, 48],
    "CAM": [244, 284, 0, 30],
    "CAR": [280, 310, 2, 29],
    "CAS": [224, 248, 32, 57],
    "COT": [115, 132, -2, 15],
    "HAL": [113, 136, -7, 17],
    "HEL": [10, 43, 24, 47],
    "HIM": [68, 103, 20, 41],
    "HIN": [62, 79, 29, 44],
    "IZU": [131, 151, 7, 38],
    "KER": [166, 191, -47, -11],
    "KUR": [118, 173, 32, 64],
    "MAK": [49, 73, 15, 36],
    "MAN": [116, 126, 10, 26],
    "MUE": [282, 303, 8, 25],
    "PAM": [64, 83, 30, 46],
    "PHI": [115, 137, -5, 24],
    "PNG": [125, 150, -12, 8],
    "PUY": [154, 175, -59, -37],
    "RYU": [113, 146, 16, 44],
    "SAM": [271, 306, -52, 18],
    "SCO": [322, 344, -65, -47],
    "SOL": [138, 167, -15, 0],
    "SUL": [112, 131, -7, 10],
    "SUM": [84, 142, -19, 34],
    "VAN": [158, 181, -31, -3],
}

regional_slab_code = regional_slab_code.upper()
if regional_slab_code not in regional_map_regions:
    raise ValueError(
        f"Unknown Slab2 region code: {regional_slab_code!r}"
    )
regional_output_directory.mkdir(parents=True, exist_ok=True)
print(
    f"Regional selection: {regional_slab_code} — "
    f"{slab_region_names[regional_slab_code]}"
)


In [ ]:
def nearest_coordinate_indices(coordinates, values):
    right = np.searchsorted(coordinates, values)
    right = np.clip(right, 1, len(coordinates) - 1)
    left = right - 1
    use_right = (
        np.abs(coordinates[right] - values)
        < np.abs(values - coordinates[left])
    )
    return np.where(use_right, right, left)


def events_within_slab_grid(events, slab_grid):
    latitude_name, longitude_name = slab_grid.dims
    latitudes = np.asarray(slab_grid[latitude_name].values)
    longitudes = np.asarray(slab_grid[longitude_name].values) % 360.0
    latitude_order = np.argsort(latitudes)
    longitude_order = np.argsort(longitudes)
    latitudes = latitudes[latitude_order]
    longitudes = longitudes[longitude_order]
    slab_values = slab_grid.values[np.ix_(latitude_order, longitude_order)]

    event_longitudes = events["longitude"] % 360.0
    event_latitudes = events["latitude"]
    inside_bounds = (
        (event_longitudes >= longitudes[0])
        & (event_longitudes <= longitudes[-1])
        & (event_latitudes >= latitudes[0])
        & (event_latitudes <= latitudes[-1])
    )
    candidate_indices = np.flatnonzero(inside_bounds)
    if not candidate_indices.size:
        return events[:0]

    longitude_indices = nearest_coordinate_indices(
        longitudes, event_longitudes[candidate_indices]
    )
    latitude_indices = nearest_coordinate_indices(
        latitudes, event_latitudes[candidate_indices]
    )
    valid_footprint = np.isfinite(
        slab_values[latitude_indices, longitude_indices]
    )
    return events[candidate_indices[valid_footprint]]


def regional_grid_spacing(region):
    maximum_span = max(region[1] - region[0], region[3] - region[2])
    if maximum_span <= 18:
        return (5, 5), (2.5, 2.5)
    if maximum_span <= 35:
        return (10, 10), (5, 5)
    if maximum_span <= 60:
        return (20, 20), (10, 10)
    return (30, 30), (15, 15)


regional_map_region = regional_map_regions[regional_slab_code]
regional_projection_center = 0.5 * (
    regional_map_region[0] + regional_map_region[1]
)
regional_major_spacing, regional_minor_spacing = regional_grid_spacing(
    regional_map_region
)
regional_slab_path = next(
    path
    for path in slab_depth_files
    if path.name.split("_slab2_", maxsplit=1)[0].upper()
    == regional_slab_code
)
regional_slab_grid = load_slab2_depth(regional_slab_path)
regional_intermediate = events_within_slab_grid(
    intermediate, regional_slab_grid
)
regional_deep_focus = events_within_slab_grid(
    deep_focus, regional_slab_grid
)

regional_relief = load_earth_relief(
    map_region, slab_data_directory, resolution="10m",
    download=download_missing_data,
)
regional_seafloor_age = load_seafloor_age(
    map_region, slab_data_directory, resolution="10m",
    download=download_missing_data,
)
regional_boundaries = load_plate_boundaries(
    slab_data_directory, download=download_missing_data,
    timeout=network_timeout_s,
)
regional_maximum_depth = max(
    100,
    math.ceil(
        max(
            float(load_slab2_depth(path).max())
            for path in slab_depth_files
        ) / 100
    ) * 100,
)
regional_slab_normalization = Normalize(
    vmin=0, vmax=regional_maximum_depth
)

regional_name = slab_region_names[regional_slab_code]
configure_matplotlib_plot(regional_slab_font_scale)
regional_slab_figure, regional_slab_axis, regional_transform = (
    create_global_map(
        f"{regional_name} ({regional_slab_code}) Slab2 depth",
        regional_relief, regional_boundaries,
        seafloor_age=regional_seafloor_age,
        display_region=regional_map_region,
        projection_center=regional_projection_center,
        major_spacing=regional_major_spacing,
        minor_spacing=regional_minor_spacing,
    )
)
regional_latitude_name, regional_longitude_name = regional_slab_grid.dims
regional_slab_axis.pcolormesh(
    regional_slab_grid[regional_longitude_name],
    regional_slab_grid[regional_latitude_name],
    regional_slab_grid.values, cmap=cmc.batlow,
    norm=regional_slab_normalization, shading="auto",
    transform=regional_transform, rasterized=True, zorder=3,
)
regional_slab_figure.subplots_adjust(bottom=0.29)
regional_slab_colorbar_axis = regional_slab_figure.add_axes(
    [0.28, 0.16, 0.48, 0.025]
)
regional_slab_colorbar = regional_slab_figure.colorbar(
    ScalarMappable(norm=regional_slab_normalization, cmap=cmc.batlow),
    cax=regional_slab_colorbar_axis, orientation="horizontal",
)
regional_slab_colorbar.set_label("Slab surface depth (km)")
regional_age_colorbar_axis = regional_slab_figure.add_axes(
    [0.28, 0.055, 0.48, 0.025]
)
regional_age_colorbar = regional_slab_figure.colorbar(
    ScalarMappable(norm=Normalize(vmin=0, vmax=180), cmap="gray_r"),
    cax=regional_age_colorbar_axis, orientation="horizontal",
)
regional_age_colorbar.set_label("Seafloor age (Ma)")
for suffix in ("png", "pdf"):
    regional_slab_output = regional_output_directory / (
        f"{regional_slab_code.lower()}_slab2_depth.{suffix}"
    )
    regional_slab_figure.savefig(
        regional_slab_output, dpi=300, bbox_inches="tight"
    )
    print(f"Saved: {regional_slab_output}")
plt.show()

configure_matplotlib_plot(regional_earthquake_font_scale)
regional_earthquake_figure, regional_earthquake_axis, regional_transform = (
    create_global_map(
        f"{regional_name} ({regional_slab_code}) GCMT earthquakes, 1976–2025",
        regional_relief, regional_boundaries, boundary_zorder=2,
        display_region=regional_map_region,
        projection_center=regional_projection_center,
        major_spacing=regional_major_spacing,
        minor_spacing=regional_minor_spacing,
    )
)
regional_earthquake_axis.scatter(
    regional_intermediate["longitude"],
    regional_intermediate["latitude"],
    s=marker_areas(regional_intermediate["moment_dyne_cm"]),
    color="#2878b5", alpha=0.65, linewidths=0,
    transform=regional_transform, zorder=3,
)
regional_earthquake_axis.scatter(
    regional_deep_focus["longitude"], regional_deep_focus["latitude"],
    s=marker_areas(regional_deep_focus["moment_dyne_cm"]),
    color="#d73027", alpha=0.7, linewidths=0,
    transform=regional_transform, zorder=3,
)
regional_depth_legend_handles = [
    Line2D(
        [], [], marker="o", linestyle="none",
        markersize=6 * plot_element_scaling_factor,
        markerfacecolor="#2878b5", markeredgewidth=0,
        label="70–300 km",
    ),
    Line2D(
        [], [], marker="o", linestyle="none",
        markersize=6 * plot_element_scaling_factor,
        markerfacecolor="#d73027", markeredgewidth=0, label="≥300 km",
    ),
]
regional_depth_legend = regional_earthquake_axis.legend(
    handles=regional_depth_legend_handles, loc="lower left", frameon=True,
)
regional_earthquake_axis.add_artist(regional_depth_legend)
regional_legend_moments = 10.0 ** (1.5 * legend_magnitudes + 16.1)
regional_magnitude_legend_handles = [
    Line2D(
        [], [], marker="o", linestyle="none",
        markersize=float(np.sqrt(area)), markerfacecolor="0.35",
        markeredgewidth=0, label=f"$M_w$ {magnitude:.1f}",
    )
    for magnitude, area in zip(
        legend_magnitudes, marker_areas(regional_legend_moments), strict=True,
    )
]
regional_earthquake_axis.legend(
    handles=regional_magnitude_legend_handles, title="Marker size",
    loc="lower right", frameon=True, labelspacing=1.0,
)
for suffix in ("png", "pdf"):
    regional_earthquake_output = regional_output_directory / (
        f"{regional_slab_code.lower()}_deep_earthquakes.{suffix}"
    )
    regional_earthquake_figure.savefig(
        regional_earthquake_output, dpi=300, bbox_inches="tight"
    )
    print(f"Saved: {regional_earthquake_output}")
print(
    f"Regional events: {regional_intermediate.size:,} intermediate-depth, "
    f"{regional_deep_focus.size:,} deep-focus"
)
plt.show()
